In [36]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import re

import matplotlib

import json
from decimal import Decimal, getcontext
from biotite.sequence.phylo import upgma, neighbor_joining

In [12]:
getcontext().prec = 4

In [13]:
with open("RXLR_WY_tmscore.json") as f:
    data = json.load(f)

# Build distance matrix

In [17]:
tm_scores = {}

for result in data["results"]:
    chain1 = result["1"]
    chain2 = result["2"]
    tm = (Decimal(result["tm1"]) + Decimal(result["tm2"])) / 2
    
    s = tuple(sorted((chain1, chain2)))
    if s in tm_scores:
        try:
            assert tm == tm_scores[s]
        except AssertionError:
            tm_scores[s] = (tm_scores[s] + tm) / 2
        
    else:
        tm_scores[s] = tm

In [18]:
df = pd.read_csv("../RXLR_WY.tsv", sep = "\t")
order = df["entry"]

In [19]:
sim_matrix =  np.identity(len(order))

In [20]:
for i, pdb1 in enumerate(order):
    for j, pdb2 in enumerate(order):
        if i == j:
            continue
        else:
            s = tuple(sorted((pdb1, pdb2)))
            tm_score = tm_scores[s]
            sim_matrix[i][j] = tm_score

In [21]:
sim_matrix

array([[1.    , 0.3104, 0.3109, ..., 0.2494, 0.324 , 0.338 ],
       [0.3104, 1.    , 0.331 , ..., 0.2094, 0.3384, 0.3492],
       [0.3109, 0.331 , 1.    , ..., 0.2502, 0.2553, 0.2636],
       ...,
       [0.2494, 0.2094, 0.2502, ..., 1.    , 0.2698, 0.2422],
       [0.324 , 0.3384, 0.2553, ..., 0.2698, 1.    , 0.8335],
       [0.338 , 0.3492, 0.2636, ..., 0.2422, 0.8335, 1.    ]],
      shape=(1802, 1802))

In [22]:
distance = 1 - sim_matrix

# Compute UPGMA tree

In [25]:
tree = upgma(distance)

In [32]:
newick_content = tree.to_newick(include_distance=True)

# ID mapping

In [33]:
id_to_gene = dict(zip(df.index, df["entry"]))

In [34]:
def replace_ids_with_genes(match):
    t1, st, t2 = match.group(0)[0], match.group(0)[1:-1], match.group(0)[-1]
    node_id = int(st)
    
    return f"{t1}{id_to_gene[node_id]}{t2}"

In [37]:
newick_content_updated = re.sub(r'\D\d+:', replace_ids_with_genes, newick_content)

# 결과를 새로운 Newick 파일로 저장
output_file = 'RXLR_structure_NJ_tree_id_updated.nwk'  # 변경: 출력 파일 경로
with open(output_file, 'w') as file:
    file.write(newick_content_updated)

In [17]:
with open("RXLR_WY_structure_upgma.nwk", "w") as f:
    f.write(tree.to_newick(include_distance=True))

In [70]:
color_strip1 = {1: "#0f0f70", 2:"#0000ff", 
                
               }

In [72]:
with open("color_strip1.dataset", "w" ) as f:
    for i, data in enumerate(zip(df["entry"], df["sup_cluster"])):
        cluster = str(data[1])

        if data[1] == 1:
            color = "#0f0f70"
        elif data[1] == -1:
            continue
        else:
            color = "#b9b9b9"
        f.write('\t'.join([str(i), color, cluster]) + "\n")

In [73]:
exp = pd.read_csv("/home/junhyeong/work/project/suppl/Experimentally_validated_effectors.csv",)

In [55]:
temp = pd.merge(df.reset_index(), exp, left_on = "entry", right_on= "id")

In [64]:
temp2 = temp[["index", "Gene_name"]]

In [65]:
temp2["Position"] = 30
temp2["Color"] = "#000000"
temp2["Style"] = "normal"
temp2["Size factor"] = 100
temp2["Rotation"] = 0

temp2.to_csv("text_label.dataset", sep = "\t", header = False, index = False)

/tmp/ipykernel_23542/624355597.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  temp2["Position"] = 30
/tmp/ipykernel_23542/624355597.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  temp2["Color"] = "#000000"
/tmp/ipykernel_23542/624355597.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#re